[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-04-retries-caching.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Retries, Timeouts, and Caching with `cache_key_fn`
**certified-journeys / prefect-certified** · Practice · Prefect for Data Engineers

> **Goal for today:** By the end of this notebook you can make tasks resilient with retries and timeouts, eliminate redundant work with result caching using `task_input_hash`, and write custom cache key functions tied to external state like file modification times.


In [ ]:
%pip install -q "prefect>=2.14" httpx


## Step 1 · How Prefect retries work

Retries are configured directly on the `@task` decorator. When a task raises any exception, Prefect:
1. Marks the run `Retrying`
2. Waits `retry_delay_seconds` (optionally with exponential back-off)
3. Re-invokes the function up to `retries` more times
4. If all attempts fail the run is marked `Failed`

```
@task(retries=3, retry_delay_seconds=5)
def fragile_task():
    ...
```

| Parameter | Type | Purpose |
|---|---|---|
| `retries` | `int` | Maximum additional attempts after the first failure |
| `retry_delay_seconds` | `int \| float \| list` | Fixed wait, or a list of delays per attempt |
| `retry_jitter_factor` | `float` | Adds randomness to avoid thundering herds |


In [ ]:
import random
from prefect import flow, task

# Track attempt number across calls in this demo
_attempt_counter: dict[str, int] = {}


@task(retries=3, retry_delay_seconds=1)  # short delay so demo runs quickly
def fetch_unreliable_data(endpoint: str) -> dict:
    """Simulates a flaky HTTP call that fails ~60 % of the time."""
    _attempt_counter[endpoint] = _attempt_counter.get(endpoint, 0) + 1
    attempt = _attempt_counter[endpoint]
    print(f"  attempt {attempt} for '{endpoint}'")
    # Succeed on the 3rd attempt so we always see at least two retries
    if attempt < 3:
        raise ConnectionError(f"Transient network error on attempt {attempt}")
    return {"endpoint": endpoint, "data": [1, 2, 3], "attempt": attempt}


@flow(log_prints=True)
def retry_demo_flow():
    result = fetch_unreliable_data("api/metrics")
    print(f"  SUCCESS: {result}")
    return result


# Run without a server — results go to the local ephemeral API
retry_demo_flow()


### What just happened?

- The task failed on attempts 1 and 2, then **succeeded on attempt 3** — exactly as configured.
- Prefect waited `retry_delay_seconds=1` between each attempt.
- **Retries are transparent to the parent flow** — the flow only sees the final successful return value.
- Setting `retry_delay_seconds` to a list like `[1, 5, 30]` gives per-attempt back-off.


## Step 2 · `timeout_seconds` and `TaskRunTimeout`

`timeout_seconds` puts a wall-clock limit on a single task attempt. When the limit is exceeded Prefect raises `prefect.exceptions.TaskRunTimeoutError` (a subclass of `TimeoutError`).  
The task is marked **`TimedOut`** — a terminal failed state — and retries (if any) will each get their own timeout budget.

```python
@task(timeout_seconds=5)
def slow_query():
    ...
```

> Timeout is enforced via a background thread signal; CPU-bound tight loops may not respect it immediately.


In [ ]:
import time
from prefect.exceptions import TaskRunTimeoutError


@task(timeout_seconds=2, retries=0)  # fail fast — no retries here
def slow_aggregation(rows: int) -> int:
    """Simulates a long database aggregation."""
    print(f"  Aggregating {rows} rows...")
    time.sleep(10)  # intentionally exceeds 2-second timeout
    return rows * 42


@flow(log_prints=True)
def timeout_demo_flow():
    # Use return_state=True so we can inspect the failure instead of raising
    state = slow_aggregation(1_000_000, return_state=True)
    print(f"  Task state type : {type(state).__name__}")
    print(f"  State name      : {state.name}")
    if state.is_failed():
        # Extract the underlying exception from the state
        exc = state.result(raise_on_failure=False)
        print(f"  Exception type  : {type(exc).__name__}")
    return state.name


timeout_demo_flow()


### What just happened?

- The task slept for 10 s but had a 2 s budget → Prefect raised `TaskRunTimeoutError`.
- **`return_state=True`** let the flow continue and inspect the state rather than propagating the exception.
- The state name is `"TimedOut"` — a distinct terminal state separate from `"Failed"`.
- **Production pattern:** combine `timeout_seconds` with `retries` — each retry resets the clock.


## Step 3 · Result caching with `task_input_hash`

Prefect caches task results keyed on a *cache key*. `task_input_hash` is the built-in function that hashes all of the task's arguments to produce that key.

```python
from prefect.tasks import task_input_hash
from datetime import timedelta

@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def expensive_fetch(url: str) -> dict:
    ...
```

| Field | Meaning |
|---|---|
| `cache_key_fn` | Callable `(ctx, params) → str` that produces the key |
| `task_input_hash` | Built-in: SHA-256 of task name + all serialised arguments |
| `cache_expiration` | `timedelta` — TTL after which the cache entry is stale |

If the cache is hit the task transitions directly to **`Cached`** and returns the stored result — **no code runs**.


In [ ]:
from datetime import timedelta
from prefect.tasks import task_input_hash

# Call counter — if caching works, this stays at 1 on the second call
_transform_calls = 0


@task(
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(minutes=5),
)
def transform_dataset(dataset_id: str, version: int) -> list:
    """Expensive transformation — should run at most once per (dataset_id, version) pair."""
    global _transform_calls
    _transform_calls += 1
    print(f"  [RUNNING] transform_dataset call #{_transform_calls} for ({dataset_id!r}, v{version})")
    # Simulate heavy work
    time.sleep(0.3)
    return [f"{dataset_id}_{version}_row_{i}" for i in range(5)]


@flow(log_prints=True)
def caching_demo_flow():
    print("--- First call (cache miss expected) ---")
    r1 = transform_dataset("sales", 3)
    print(f"  Result: {r1}")

    print("--- Second call, same args (cache hit expected) ---")
    r2 = transform_dataset("sales", 3)
    print(f"  Result: {r2}")

    print("--- Third call, different version (cache miss expected) ---")
    r3 = transform_dataset("sales", 4)
    print(f"  Result: {r3}")

    print(f"\n  Total actual task executions: {_transform_calls} (expected 2)")


caching_demo_flow()


### What just happened?

- The first call computed and stored the result under a hash of `("sales", 3)`.
- The second call with identical arguments was a **cache hit** — the task body did not run.
- The third call with `version=4` produced a **different hash** → cache miss → task ran again.
- **`_transform_calls` was 2, not 3** — caching eliminated one redundant execution.


## Step 4 · Custom `cache_key_fn` using file modification time

Sometimes task inputs are not enough to describe freshness — a task might depend on a file that changes independently. A **custom `cache_key_fn`** lets you incorporate any external state into the key.

The signature is:
```python
def my_key(context: TaskRunContext, parameters: dict) -> str:
    ...
    return unique_string
```

`context` gives you access to `context.task.name`, `context.task_run`, etc.  
`parameters` is the raw kwarg dict passed to the task.


In [ ]:
import hashlib
import os
import pathlib
from prefect.context import TaskRunContext

# Create a demo file to track
DEMO_FILE = pathlib.Path("/tmp/prefect_demo_config.json")
DEMO_FILE.write_text('{"threshold": 0.9}')


def file_mtime_cache_key(context: TaskRunContext, parameters: dict) -> str:
    """
    Cache key = hash of (task name, all parameters, file modification time).
    When the file is updated the mtime changes, invalidating the cache.
    """
    file_path: str = parameters.get("config_path", "")
    try:
        mtime = os.path.getmtime(file_path)
    except FileNotFoundError:
        mtime = 0.0

    raw = f"{context.task.name}:{parameters}:{mtime}"
    return hashlib.sha256(raw.encode()).hexdigest()


_load_calls = 0


@task(cache_key_fn=file_mtime_cache_key, cache_expiration=timedelta(hours=24))
def load_config(config_path: str) -> dict:
    """Loads and parses a JSON config — skipped if the file hasn't changed."""
    import json
    global _load_calls
    _load_calls += 1
    print(f"  [RUNNING] load_config #{_load_calls}, reading {config_path}")
    return json.loads(pathlib.Path(config_path).read_text())


@flow(log_prints=True)
def custom_cache_key_flow():
    print("--- Call 1: file unchanged ---")
    c1 = load_config(str(DEMO_FILE))
    print(f"  config: {c1}")

    print("--- Call 2: same file, same mtime → cache hit ---")
    c2 = load_config(str(DEMO_FILE))
    print(f"  config: {c2}")

    # Touch the file to simulate an update
    time.sleep(0.01)  # ensure mtime changes on fast filesystems
    DEMO_FILE.write_text('{"threshold": 0.75}')  # new content → new mtime

    print("--- Call 3: file updated → cache miss ---")
    c3 = load_config(str(DEMO_FILE))
    print(f"  config: {c3}")

    print(f"\n  Total load_config executions: {_load_calls} (expected 2)")


custom_cache_key_flow()


### What just happened?

- Our custom key function embeds the file's `mtime` — so **modifying the file busts the cache**.
- Calls 1 and 3 actually ran; call 2 was served from cache.
- **The pattern generalises**: hash a database watermark, an S3 ETag, or any version signal.
- `context.task.name` is included in the key to prevent collisions between different tasks that share the same parameters.


## Step 5 · `cache_expiration` — TTL-based cache invalidation

`cache_expiration` accepts a `timedelta`. Once the TTL elapses, the cached result is considered stale and the task runs again on the next call.  

Common patterns:

| Scenario | Suggested expiration |
|---|---|
| Hourly batch pipeline | `timedelta(hours=1)` |
| Daily report | `timedelta(hours=23)` |
| Expensive but rarely-changing reference data | `timedelta(days=7)` |
| Development / debugging | `timedelta(seconds=30)` |


In [ ]:
from datetime import timedelta

_expiry_calls = 0


@task(
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(seconds=2),  # very short TTL for demo
)
def fetch_reference_rates(currency: str) -> float:
    """Fetches an FX rate — expensive, but data refreshes every few seconds."""
    global _expiry_calls
    _expiry_calls += 1
    import random
    rate = round(1.0 + random.uniform(-0.05, 0.05), 4)
    print(f"  [RUNNING] fetch_reference_rates #{_expiry_calls} → {rate}")
    return rate


@flow(log_prints=True)
def cache_expiry_flow():
    print("--- Run 1: cache miss (first call) ---")
    r1 = fetch_reference_rates("EUR")
    print(f"  rate: {r1}")

    print("--- Run 2: within TTL → cache hit ---")
    r2 = fetch_reference_rates("EUR")
    print(f"  rate: {r2} (same as above? {r1 == r2})")

    # Wait for cache to expire
    print("  [waiting 3 s for TTL to expire...]")
    time.sleep(3)

    print("--- Run 3: TTL expired → cache miss, new value ---")
    r3 = fetch_reference_rates("EUR")
    print(f"  rate: {r3}")

    print(f"\n  Total fetch executions: {_expiry_calls} (expected 2)")


cache_expiry_flow()


### What just happened?

- Run 2 returned the **same value** as run 1 because the 2-second TTL had not elapsed.
- After `time.sleep(3)`, the cache entry was stale → run 3 re-executed the task body.
- `_expiry_calls` was 2, confirming one cache hit.
- **In production** you'd set `cache_expiration` to match how often upstream data actually changes.


## Step 6 · Combining retries + timeout + caching

All three parameters compose cleanly. A good production pattern for a flaky external API call:

```python
@task(
    retries=3,
    retry_delay_seconds=[2, 10, 30],   # exponential-style back-off
    timeout_seconds=15,                 # give up if one attempt hangs
    cache_key_fn=task_input_hash,       # skip if already computed today
    cache_expiration=timedelta(hours=6),
)
def fetch_from_api(query: str) -> dict:
    ...
```

The cache is checked **before** the first attempt — if there's a valid cache hit, no network call is made and no retry budget is consumed.


In [ ]:
_api_attempts = []


@task(
    retries=2,
    retry_delay_seconds=1,
    timeout_seconds=5,
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(minutes=10),
)
def resilient_api_call(query: str) -> dict:
    """A task that is both fault-tolerant and cached."""
    _api_attempts.append(query)
    attempt_num = len([q for q in _api_attempts if q == query])
    print(f"  [RUNNING] attempt {attempt_num} for query={query!r}")
    if attempt_num == 1:  # first attempt always fails
        raise ConnectionError("first-attempt failure")
    # Simulate a real response
    return {"query": query, "rows": 42, "attempt": attempt_num}


@flow(log_prints=True)
def combined_resilience_flow():
    print("--- First invocation (retry expected on first attempt) ---")
    result1 = resilient_api_call("SELECT * FROM sales")
    print(f"  Result: {result1}")

    print("--- Second invocation (cache hit — zero retries consumed) ---")
    result2 = resilient_api_call("SELECT * FROM sales")
    print(f"  Result: {result2}")
    print(f"  Total actual executions: {len(_api_attempts)}")


combined_resilience_flow()


### What just happened?

- The first call failed once, then succeeded on the retry — total 2 task-body executions.
- The second call was a **cache hit** — zero executions, zero retries consumed.
- **Caching amplifies the value of retries**: once a hard-won result is stored, subsequent calls get it free.
- The `timeout_seconds=5` guard ensures neither attempt hangs indefinitely.


In [ ]:
# ─── Challenge ────────────────────────────────────────────────────────────────
# Challenge: Write a cache key function that incorporates BOTH the task's
# input arguments AND an environment variable (e.g. DATA_VERSION).
# If DATA_VERSION is not set, fall back to "v1".
#
# Then decorate a task `score_model(dataset: str) -> float` with:
#   - retries=2, retry_delay_seconds=1
#   - timeout_seconds=10
#   - your custom cache_key_fn
#   - cache_expiration=timedelta(hours=12)
#
# Verify the cache busts when you change the DATA_VERSION env var.
# ──────────────────────────────────────────────────────────────────────────────

import os
import hashlib
from prefect.context import TaskRunContext


def env_aware_cache_key(context: TaskRunContext, parameters: dict) -> str:
    # TODO: read DATA_VERSION from env, default to "v1"
    # TODO: build and return a hash string from task name + params + version
    pass


@task(
    # TODO: add retries, timeout, cache_key_fn, cache_expiration
)
def score_model(dataset: str) -> float:
    # TODO: simulate scoring (return a float)
    pass


@flow(log_prints=True)
def challenge_flow():
    # TODO: call score_model twice with the same dataset (expect cache hit)
    # TODO: change DATA_VERSION env var and call again (expect cache miss)
    pass


# challenge_flow()


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `retries=N` | Up to N additional attempts after the first failure |
| `retry_delay_seconds` | Fixed int, float, or list for per-attempt delays |
| `timeout_seconds` | Per-attempt wall-clock limit; exceeded → `TimedOut` state |
| `task_input_hash` | Hashes all arguments; identical inputs → cache hit |
| Custom `cache_key_fn` | `(ctx, params) → str` — embed any external state in the key |
| `cache_expiration` | `timedelta` TTL; stale entries trigger a re-run |
| Cache + retries | Cache is checked first; a hit skips all retry logic |

> **Tip:** `task_input_hash` hashes all task arguments — if inputs haven't changed, the task is skipped entirely and returns the cached result. Use it aggressively for expensive data fetching steps.

---
## What's next
**Day 5** → Subflows and Modular Pipeline Design — break large pipelines into independently-runnable, testable subflows.

Mark Day 4 complete in your [tracker](../index.html).
